# Refrigerant Screening Workflow

This notebook demonstrates a complete screening workflow for identifying refrigerant candidates using PC-SAFT parameter predictions.

**Use Case**: Screen fluorinated hydrocarbons for similarity to **R-134a** (1,1,1,2-tetrafluoroethane), a common refrigerant being phased out due to high GWP.

> **Important**: This is an **illustrative workflow**, not a validated industrial standard. Predictions are screening-quality (R² ~ 0.73-0.77) and should be validated experimentally before synthesis or deployment.

**What you'll learn**:
1. Define a reference compound (R-134a)
2. Generate candidate molecules via systematic enumeration
3. Predict PC-SAFT parameters with uncertainty
4. Filter by applicability domain and uncertainty thresholds
5. Rank by parameter similarity to reference
6. Visualize and export top candidates

**Runtime**: ~2-3 minutes for 200 candidates

## 1. Setup and Reference Compound

Define R-134a as our reference compound:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

import pcsaft_predict

# Reference compound: R-134a (1,1,1,2-tetrafluoroethane)
reference = {
    "name": "R-134a",
    "smiles": "FC(F)(F)CF",
    "cas": "811-97-2",
    "mw": 102.03,
    "bp_K": 247.1,  # -26°C
    "gwp": 1430  # 100-year GWP
}

print(f"Reference Compound: {reference['name']}")
print(f"  SMILES: {reference['smiles']}")
print(f"  Molecular Weight: {reference['mw']:.2f} g/mol")
bp = reference["bp_K"]
print(f"  Boiling Point: {bp:.1f} K ({bp - 273.15:.1f} C)")
print(f"  GWP (100y): {reference['gwp']}")

# Predict PC-SAFT parameters for reference
df_ref = pcsaft_predict.predict_with_uncertainty(reference["smiles"])
ref_params = {
    "m": df_ref["m"].iloc[0],
    "sigma": df_ref["sigma"].iloc[0],
    "epsilon_k": df_ref["epsilon_k"].iloc[0]
}

name = reference["name"]
print(f"\nPredicted PC-SAFT Parameters for {name}:")
m_std = df_ref["m_std"].iloc[0]
s_std = df_ref["sigma_std"].iloc[0]
e_std = df_ref["epsilon_k_std"].iloc[0]
print(f"  m: {ref_params['m']:.3f} +/- {m_std:.3f}")
print(f"  sigma: {ref_params['sigma']:.3f} +/- {s_std:.3f}")
print(f"  eps/k: {ref_params['epsilon_k']:.1f} +/- {e_std:.1f}")
tnn = df_ref["tanimoto_nn"].iloc[0]
print(f"  In domain: {df_ref['in_domain'].iloc[0]} (Tanimoto: {tnn:.3f})")

## 2. Generate Candidate Molecules

Create a diverse set of fluorinated candidates via systematic enumeration:

**Strategy**:
1. C2-C5 alkanes and alkenes
2. Substitute H with F (0-6 fluorines)
3. Include both saturated (HFCs) and unsaturated (HFOs)
4. Filter for chemical validity

In [ ]:
from itertools import combinations


def generate_fluorinated_candidates(max_carbons=5, max_fluorines=6):
    """Generate fluorinated hydrocarbon candidates."""
    candidates = set()

    # Base structures (C2-C5 alkanes and alkenes)
    backbones = [
        "CC",       # ethane
        "C=C",      # ethene
        "CCC",      # propane
        "C=CC",     # propene
        "CCCC",     # butane
        "C=CCC",    # 1-butene
        "CC=CC",    # 2-butene
        "CCCCC",    # pentane
        "C=CCCC",   # 1-pentene
    ]

    for backbone in backbones:
        mol = Chem.MolFromSmiles(backbone)
        if mol is None:
            continue

        # Count hydrogens
        mol_h = Chem.AddHs(mol)
        h_atoms = [
            atom.GetIdx()
            for atom in mol_h.GetAtoms()
            if atom.GetSymbol() == "H"
        ]

        # Generate F-substitution patterns
        n_max = min(len(h_atoms), max_fluorines) + 1
        for n_fluorines in range(n_max):
            if n_fluorines == 0:
                candidates.add(Chem.MolToSmiles(mol))
                continue

            import random
            all_combos = list(combinations(h_atoms, n_fluorines))
            sample_size = min(20, len(all_combos))
            sampled_combos = random.sample(all_combos, sample_size)

            for h_indices in sampled_combos:
                mol_f = Chem.RWMol(mol_h)
                for h_idx in sorted(h_indices, reverse=True):
                    atom = mol_f.GetAtomWithIdx(h_idx)
                    if atom.GetSymbol() == "H":
                        mol_f.RemoveAtom(h_idx)

    # Pre-defined set of common refrigerants and analogs
    candidate_smiles = [
        # HFCs (saturated)
        "FC(F)(F)C(F)(F)F",  # R-116
        "FC(F)C(F)(F)F",     # R-125
        "FC(F)(F)CF",        # R-134a (reference)
        "FCC(F)(F)F",        # R-143a
        "FC(F)CF",           # R-143
        "FCCC(F)(F)F",       # R-245fa analog
        "FC(F)(F)C(F)F",     # R-125 isomer
        # HFOs (unsaturated)
        "FC(F)=CF",          # R-1123
        "FC(=C)F",           # R-1132a
        "FC(F)=C(F)F",       # R-1114
        "FC(=CF)C(F)(F)F",   # R-1234yf analog
        "FC(F)=CC(F)(F)F",   # R-1234ze(E)
        "FC(F)=CCF",         # HFO analog
        "FC=CC(F)(F)F",      # HFO analog
        "FC(F)=CCC",         # HFO analog
        "C=C(F)C(F)(F)F",    # HFO analog
        # C3 compounds
        "FC(F)(F)CC", "FCC(F)C", "FC(F)CC", "FCCC",
        "FC(F)(F)C=C", "FC=CC", "C=CC(F)(F)F",
        # C4 compounds
        "FC(F)(F)CCC", "FCCCC", "FC(F)CCC",
        "FC(F)(F)C=CC", "FC=CCC",
    ]

    # Expand with substituted variants
    for base in ["CC", "CCC", "CCCC", "C=CC", "C=CCC"]:
        for pattern in ["F", "F(F)", "F(F)(F)"]:
            try:
                smiles = base.replace("C", f"C{pattern}", 1)
                mol = Chem.MolFromSmiles(smiles)
                if mol is not None:
                    candidate_smiles.append(Chem.MolToSmiles(mol))
            except Exception:
                pass

    # Canonicalize and deduplicate
    valid_smiles = set()
    for smi in candidate_smiles:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            try:
                Chem.SanitizeMol(mol)
                valid_smiles.add(Chem.MolToSmiles(mol))
            except Exception:
                pass

    return list(valid_smiles)


# Generate candidates
candidates_smiles = generate_fluorinated_candidates()
print(f"Generated {len(candidates_smiles)} unique candidates")

# Preview first 10
print("\nSample candidates:")
for i, smi in enumerate(candidates_smiles[:10], 1):
    mol = Chem.MolFromSmiles(smi)
    mw = Descriptors.MolWt(mol)
    n_f = smi.count("F")
    print(f"  {i:2d}. {smi:20s} (MW={mw:6.2f}, F={n_f})")

## 3. Predict PC-SAFT Parameters

Run batch prediction with uncertainty quantification:

In [ ]:
n_cand = len(candidates_smiles)
print(f"Predicting PC-SAFT parameters for {n_cand} candidates...")

# Batch prediction
df_candidates = pcsaft_predict.predict_with_uncertainty(candidates_smiles)

# Add molecular descriptors
mws = []
n_fluorines = []
has_double_bond = []

for smi in df_candidates["smiles"]:
    mol = Chem.MolFromSmiles(smi)
    mws.append(Descriptors.MolWt(mol) if mol else np.nan)
    n_fluorines.append(smi.count("F"))
    has_double_bond.append("=" in smi)

df_candidates["mw"] = mws
df_candidates["n_fluorines"] = n_fluorines
df_candidates["has_double_bond"] = has_double_bond
df_candidates["molecule_class"] = df_candidates["has_double_bond"].apply(
    lambda x: "HFO" if x else "HFC"
)

n_valid = df_candidates["m"].notna().sum()
n_total = len(df_candidates)
print(f"\nPrediction complete. Valid: {n_valid}/{n_total}")
print("\nMolecule class distribution:")
print(df_candidates["molecule_class"].value_counts())

## 4. Filter by Applicability Domain and Uncertainty

Apply screening criteria to identify reliable predictions:

In [ ]:
# Define screening criteria
criteria = {
    "valid_prediction": df_candidates["m"].notna(),
    "acceptable_uncertainty": df_candidates["epsilon_k_std"] < 20.0,
    "tanimoto_threshold": df_candidates["tanimoto_nn"] >= 0.3,
    "mw_range": (df_candidates["mw"] >= 50) & (df_candidates["mw"] <= 200),
}

print("Screening Criteria Pass Rates:")
print("=" * 60)
n_total = len(df_candidates)
for criterion, mask in criteria.items():
    n_pass = mask.sum()
    pct = 100 * n_pass / n_total
    print(f"  {criterion:25s}: {n_pass:3d}/{n_total:3d} ({pct:5.1f}%)")

# Combined filter
all_criteria = pd.Series(True, index=df_candidates.index)
for mask in criteria.values():
    all_criteria &= mask

df_filtered = df_candidates[all_criteria].copy()
n_filt = len(df_filtered)
pct_pass = 100 * n_filt / n_total
print(f"\nFinal candidates passing all criteria: {n_filt}/{n_total} ({pct_pass:.1f}%)")

if n_filt == 0:
    print("\nNo candidates passed all criteria. Relaxing constraints...")
    criteria["acceptable_uncertainty"] = df_candidates["epsilon_k_std"] < 25.0
    all_criteria = pd.Series(True, index=df_candidates.index)
    for mask in criteria.values():
        all_criteria &= mask
    df_filtered = df_candidates[all_criteria].copy()
    print(f"After relaxing to 25 K: {len(df_filtered)} candidates")

## 5. Rank by Parameter Similarity to R-134a

Compute Euclidean distance in normalized PC-SAFT parameter space:

In [ ]:
# Normalize parameters to [0, 1] for fair distance calculation
def normalize(x, x_ref, x_std):
    """Normalize by reference value and typical scale."""
    return (x - x_ref) / x_std

# Typical scales from training data
typical_scales = {
    "m": 1.5,       # Typical std dev for m
    "sigma": 0.3,   # Typical std dev for sigma
    "epsilon_k": 50  # Typical std dev for epsilon_k
}

# Compute normalized distance
df_filtered["delta_m"] = normalize(
    df_filtered["m"], ref_params["m"], typical_scales["m"]
)
df_filtered["delta_sigma"] = normalize(
    df_filtered["sigma"], ref_params["sigma"], typical_scales["sigma"]
)
df_filtered["delta_epsilon_k"] = normalize(
    df_filtered["epsilon_k"], ref_params["epsilon_k"], typical_scales["epsilon_k"]
)

df_filtered["distance"] = np.sqrt(
    df_filtered["delta_m"]**2 +
    df_filtered["delta_sigma"]**2 +
    df_filtered["delta_epsilon_k"]**2
)

# Sort by distance (lower = more similar)
df_ranked = df_filtered.sort_values("distance").reset_index(drop=True)

print(f"\nTop 10 Candidates (ranked by parameter similarity to {reference['name']}):")
print("=" * 120)
display_cols = [
    "smiles", "m", "sigma", "epsilon_k", "distance", 
    "n_fluorines", "mw", "molecule_class", "tanimoto_nn"
]
print(df_ranked[display_cols].head(10).to_string(index=True))

## 6. Visualize Top Candidates

Plot parameter space and compare to reference:

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Distance vs number of fluorines
ax = axes[0]
colors = df_ranked["molecule_class"].map(
    {"HFC": "blue", "HFO": "orange"}
)
scatter = ax.scatter(
    df_ranked["n_fluorines"],
    df_ranked["distance"],
    c=colors, s=80, alpha=0.6,
    edgecolors="black", linewidth=0.5,
)

# Highlight top 5
top5 = df_ranked.head(5)
ax.scatter(
    top5["n_fluorines"], top5["distance"],
    s=200, facecolors="none", edgecolors="red",
    linewidth=2, label="Top 5 candidates",
)

ax.axhline(
    0, color="green", linestyle="--",
    linewidth=2, alpha=0.7, label="R-134a (reference)",
)
ax.set_xlabel("Number of Fluorine Atoms", fontsize=12)
ax.set_ylabel("Normalized Parameter Distance", fontsize=12)
ax.set_title(
    "Candidate Similarity to R-134a",
    fontsize=13, fontweight="bold",
)
ax.legend()
ax.grid(alpha=0.3)

# Panel B: eps/k vs sigma parameter space
ax = axes[1]
scatter = ax.scatter(
    df_ranked["sigma"], df_ranked["epsilon_k"],
    c=colors, s=80, alpha=0.6,
    edgecolors="black", linewidth=0.5,
)

# Reference point
ax.scatter(
    ref_params["sigma"], ref_params["epsilon_k"],
    s=300, marker="*", c="red",
    edgecolors="black", linewidth=1.5,
    label="R-134a", zorder=10,
)

# Top 5
ax.scatter(
    top5["sigma"], top5["epsilon_k"],
    s=200, facecolors="none", edgecolors="red",
    linewidth=2, label="Top 5 candidates",
)

ax.set_xlabel("sigma (A)", fontsize=12)
ax.set_ylabel("eps/k (K)", fontsize=12)
ax.set_title(
    "Parameter Space (sigma vs eps/k)",
    fontsize=13, fontweight="bold",
)
ax.legend()
ax.grid(alpha=0.3)

# Legend for molecule class
legend_elements = [
    Patch(facecolor="blue", label="HFC (saturated)"),
    Patch(facecolor="orange", label="HFO (unsaturated)"),
]
fig.legend(
    handles=legend_elements, loc="lower center",
    ncol=2, bbox_to_anchor=(0.5, -0.05), fontsize=11,
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## 7. Export Top Candidates

Save results for experimental validation or further analysis:

In [ ]:
# Select top 20 candidates
top_n = 20
df_export = df_ranked.head(top_n).copy()

# Add reference comparison columns
df_export["m_ref"] = ref_params["m"]
df_export["sigma_ref"] = ref_params["sigma"]
df_export["epsilon_k_ref"] = ref_params["epsilon_k"]

m_ref = ref_params["m"]
s_ref = ref_params["sigma"]
e_ref = ref_params["epsilon_k"]
df_export["m_pct_diff"] = 100 * (df_export["m"] - m_ref) / m_ref
df_export["sigma_pct_diff"] = (
    100 * (df_export["sigma"] - s_ref) / s_ref
)
df_export["epsilon_k_pct_diff"] = (
    100 * (df_export["epsilon_k"] - e_ref) / e_ref
)

# Reorder columns for clarity
export_cols = [
    "smiles", "molecule_class", "n_fluorines", "mw",
    "m", "m_std", "m_pct_diff",
    "sigma", "sigma_std", "sigma_pct_diff",
    "epsilon_k", "epsilon_k_std", "epsilon_k_pct_diff",
    "distance", "tanimoto_nn", "in_domain"
]

df_export_final = df_export[export_cols]

# Save to CSV
output_file = "r134a_screening_results.csv"
df_export_final.to_csv(output_file, index=False)
print(f"\nExported top {top_n} candidates to {output_file}")

# Display summary
print("\nTop 5 Candidates for Experimental Validation:")
print("=" * 120)
summary_cols = [
    "smiles", "molecule_class", "n_fluorines",
    "m", "sigma", "epsilon_k", "distance",
]
print(df_export_final[summary_cols].head(5).to_string(index=True))

## 8. Summary and Caveats

**What we accomplished**:
1. ✓ Defined R-134a as reference refrigerant
2. ✓ Generated ~30-50 fluorinated candidates (HFCs + HFOs)
3. ✓ Predicted PC-SAFT parameters with uncertainty
4. ✓ Filtered by applicability domain (Tanimoto ≥ 0.3) and uncertainty (ε/k_std < 20 K)
5. ✓ Ranked by normalized Euclidean distance in parameter space
6. ✓ Exported top 20 candidates for validation

**Key caveats** (READ BEFORE USING RESULTS):

1. **This is NOT a validated workflow**: It is illustrative only. Real refrigerant screening requires:
   - Experimental VLE data to fit PC-SAFT parameters (not ML predictions)
   - Toxicity assessment (REACH, EPA SNAP approval)
   - Flammability testing (ASHRAE 34 classification)
   - Environmental impact (GWP, ODP, atmospheric lifetime)
   - Equipment compatibility (lubricant miscibility, material compatibility)

2. **Model limitations**:
   - R² ~ 0.73 for ε/k (screening-quality, not reference-quality)
   - Training data has only ~10% fluorinated compounds
   - Many candidates are out-of-domain (Tanimoto < 0.4)
   - Uncertainty underestimated for novel chemistries

3. **Parameter similarity ≠ viable replacement**:
   - Commercial refrigerants (e.g., R-1234yf) often have DIFFERENT parameters from R-134a
   - System redesign (compressor, heat exchanger, charge amount) is usually required
   - Drop-in replacements rarely exist for high-GWP refrigerants

4. **Next steps for serious use**:
   - Synthesize top 3-5 candidates
   - Measure experimental VLE data
   - Refit PC-SAFT parameters with experimental data
   - Conduct safety and environmental testing
   - Submit to regulatory approval (EPA SNAP, EU F-gas regulation)

**This workflow demonstrates the methodology, not a production-ready screening pipeline.**